## 🎯 Learning Objectives
* Understand the limitations of traditional RAG systems and the need for dynamic information retrieval.
* Learn the architectural patterns for implementing agentic RAG with web search fallback using LangGraph.
* Implement a LangGraph agent that intelligently decides between internal knowledge retrieval and external web search.
* Evaluate the trade-offs and practical considerations of integrating web search into RAG workflows.


## Agentic RAG with Web Search Fallback: Expanding Knowledge Horizons

Traditional Retrieval-Augmented Generation (RAG) systems are powerful, but they operate within the confines of their pre-indexed knowledge base. What happens when a user asks a question about a recent event, a niche topic not covered in the internal documents, or when the internal information is simply insufficient or outdated? This is where **Agentic RAG with Web Search Fallback** shines, transforming a static knowledge system into a dynamic, adaptive information powerhouse.

### The Librarian Analogy

Imagine a highly knowledgeable librarian (our RAG system) who has access to an extensive, well-organized library (your internal knowledge base). Most of the time, they can find the perfect book to answer your question. But what if you ask about the latest scientific discovery that hasn't been published in their books yet, or a very obscure fact they've never encountered? A traditional librarian would simply say, "I don't have that information." 

Now, imagine this librarian also has a super-fast internet connection and is skilled at quickly searching the web. If they can't find the answer in their books, they can seamlessly turn to the internet, find the relevant information, and then synthesize it with their existing knowledge to give you a comprehensive answer. This is precisely what Agentic RAG with Web Search Fallback achieves.

### How it Works: A Step-by-Step Flow

This advanced pattern leverages the orchestration capabilities of frameworks like LangGraph to create a multi-step, intelligent workflow:

1.  **Initial RAG Attempt**: The system first attempts to answer the query using its primary RAG mechanism, querying the internal knowledge base.
2.  **Confidence/Relevance Evaluation**: An intelligent agent (often an LLM) evaluates the quality, relevance, and completeness of the RAG-retrieved documents or the initial RAG-generated answer. It asks: "Is this information sufficient to answer the user's query accurately and comprehensively?"
3.  **Conditional Web Search**: 
    *   If the RAG output is deemed sufficient, the process proceeds to generate the final answer based on internal knowledge.
    *   If the RAG output is insufficient, outdated, or irrelevant, the agent triggers a **web search tool**. This tool queries external sources (like Google Search, Brave Search, or specialized APIs) to find up-to-date or broader information.
4.  **Information Synthesis**: Once web search results are obtained (if triggered), the agent synthesizes information from *both* the internal RAG and the external web search results. This step is crucial for providing a holistic and accurate answer, avoiding contradictions, and ensuring coherence.
5.  **Final Answer Generation**: The synthesized information is then used by the LLM to generate the final, comprehensive answer to the user's query.

### Why is this "Agentic"?

The "agentic" aspect comes from the system's ability to make autonomous decisions: to evaluate its own performance (RAG output), to choose the appropriate tool (internal RAG or web search), and to orchestrate a multi-step reasoning process to achieve its goal. LangGraph provides the perfect canvas for defining these stateful, cyclical decision-making processes, allowing us to build robust and highly capable RAG systems that are truly 2026-ready.


In [ ]:
import os
from typing import List, Dict, Any
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

# --- Configuration and API Keys (2026 Ready) ---
# Ensure you have your API keys set up as environment variables.
# For Google Gemini, use GOOGLE_API_KEY.
# For OpenAI, use OPENAI_API_KEY.
# For Anthropic Claude, use ANTHROPIC_API_KEY.
# For Tavily Search (a good web search tool), use TAVILY_API_KEY.

# Using Google Gemini 1.5 Flash for its speed and context window
# from langchain_google_genai import ChatGoogleGenerativeAI
# llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.1)

# Or using OpenAI GPT-4o for its strong reasoning capabilities
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o", temperature=0.1)

# Mock RAG Tool (simulates internal knowledge base retrieval)
@tool
def retrieve_from_internal_kb(query: str) -> str:
    """Retrieves relevant documents from the internal knowledge base based on the query."""
    # In a real scenario, this would call your vector database and retrieve chunks.
    # For demonstration, we'll simulate different outcomes.
    if "LangGraph" in query or "agentic RAG" in query:
        return "LangGraph is a library for building stateful, multi-actor applications with LLMs. It extends LangChain by adding the ability to create cyclic graphs, essential for agentic workflows like conditional routing and self-correction. Agentic RAG combines LLMs with external tools and decision-making capabilities to dynamically retrieve and synthesize information."
    elif "latest AI regulations" in query:
        return "Our internal knowledge base contains information on AI regulations up to early 2024. For the absolute latest, external search might be required."
    elif "quantum computing breakthroughs 2025" in query:
        return "No specific breakthroughs for 2025 found in our internal documents. Our data is current up to mid-2024."
    else:
        return "No highly relevant documents found in the internal knowledge base for this query. The information might be too niche or too recent."

# Web Search Tool (using Tavily Search for real-world example)
# In 2026, we'd likely use highly optimized, low-latency search APIs.
# For this example, we'll use a mock, but you'd integrate a real one like Tavily.

# from langchain_community.tools.tavily_search import TavilySearchResults
# web_search_tool = TavilySearchResults(max_results=3)

@tool
def web_search(query: str) -> str:
    """Performs a web search for the given query and returns relevant snippets."""
    # Simulate web search results based on query
    if "latest AI regulations" in query:
        return "Web search results: Recent EU AI Act updates (late 2024/early 2025) focus on high-risk AI systems and transparency. US discussions involve executive orders and sector-specific guidelines. UK is exploring a pro-innovation approach. Key themes: data privacy, bias mitigation, accountability."
    elif "quantum computing breakthroughs 2025" in query:
        return "Web search results: 2025 saw significant advancements in error correction techniques for superconducting qubits, with Google and IBM reporting sustained coherence times exceeding previous records. New algorithms for quantum chemistry simulations also emerged, promising faster drug discovery."
    elif "LangGraph" in query:
        return "Web search results: LangGraph is a LangChain extension for building robust, stateful LLM applications. It enables complex agentic workflows, conditional routing, and human-in-the-loop systems. Official documentation and tutorials are available on the LangChain website."
    else:
        return f"Web search for '{query}' yielded various results, including recent news articles and academic papers. Example snippet: '...[relevant snippet for {query}]...'"

# --- Define Graph State ---
class AgentState(Dict): # Using Dict for simplicity, Pydantic model for robustness in production
    query: str
    rag_result: str = ""
    web_search_result: str = ""
    final_answer: str = ""
    steps: List[str] = []

# --- Define Graph Nodes ---
def retrieve_node(state: AgentState) -> AgentState:
    print("---NODE: RETRIEVE FROM INTERNAL KB---")
    query = state["query"]
    rag_result = retrieve_from_internal_kb.invoke({"query": query})
    state["rag_result"] = rag_result
    state["steps"].append(f"Retrieved from KB: {rag_result[:50]}...")
    return state

def decide_to_search_node(state: AgentState) -> str:
    print("---NODE: DECIDE TO SEARCH---")
    query = state["query"]
    rag_result = state["rag_result"]

    # LLM-based decision making
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are an expert assistant. Based on the internal RAG result, decide if a web search is necessary to fully and accurately answer the user's query. Respond with 'web_search' if needed, otherwise 'generate_answer'.\n\nInternal RAG Result: {rag_result}"),
        ("human", "User Query: {query}\n\nDecision:")
    ])
    
    decision_chain = prompt | llm | RunnableLambda(lambda x: x.content.strip().lower())
    decision = decision_chain.invoke({"query": query, "rag_result": rag_result})
    
    print(f"Decision: {decision}")
    state["steps"].append(f"Decision: {decision}")
    
    if "web_search" in decision:
        return "web_search"
    else:
        return "generate_answer"

def web_search_node(state: AgentState) -> AgentState:
    print("---NODE: PERFORM WEB SEARCH---")
    query = state["query"]
    web_result = web_search.invoke({"query": query})
    state["web_search_result"] = web_result
    state["steps"].append(f"Performed web search: {web_result[:50]}...")
    return state

def generate_answer_node(state: AgentState) -> AgentState:
    print("---NODE: GENERATE FINAL ANSWER---")
    query = state["query"]
    rag_result = state["rag_result"]
    web_search_result = state["web_search_result"]

    # Synthesize information from available sources
    context = f"Internal Knowledge Base Result:\n{rag_result}\n\n"
    if web_search_result:
        context += f"Web Search Results:\n{web_search_result}\n\n"
    
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant. Based on the provided context, answer the user's query comprehensively and accurately. If information from both internal KB and web search is available, synthesize it coherently. If no relevant information is found, state that you cannot provide a definitive answer.\n\nContext:\n{context}"),
        ("human", "User Query: {query}\n\nAnswer:")
    ])
    
    answer_chain = prompt | llm | RunnableLambda(lambda x: x.content)
    final_answer = answer_chain.invoke({"query": query, "context": context})
    
    state["final_answer"] = final_answer
    state["steps"].append(f"Generated final answer: {final_answer[:50]}...")
    return state

# --- Build the LangGraph Graph ---
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("retrieve", retrieve_node)
workflow.add_node("decide_to_search", decide_to_search_node)
workflow.add_node("web_search", web_search_node)
workflow.add_node("generate_answer", generate_answer_node)

# Set entry point
workflow.set_entry_point("retrieve")

# Add edges
workflow.add_edge("retrieve", "decide_to_search")
workflow.add_conditional_edges(
    "decide_to_search",
    decide_to_search_node, # The function itself determines the next state
    {
        "web_search": "web_search",
        "generate_answer": "generate_answer"
    }
)
workflow.add_edge("web_search", "generate_answer")
workflow.add_edge("generate_answer", END)

# Compile the graph
app = workflow.compile()

# --- Run Examples ---
print("\n--- Running Example 1: Query well-covered by internal KB ---")
query1 = "What is LangGraph and how does it relate to agentic RAG?"
for s in app.stream({"query": query1, "steps": []}):
    print(s)

print("\n--- Running Example 2: Query needing web search for latest info ---")
query2 = "What were the latest AI regulations introduced in late 2024 or early 2025?"
for s in app.stream({"query": query2, "steps": []}):
    print(s)

print("\n--- Running Example 3: Query needing web search for specific recent breakthroughs ---")
query3 = "What were the major quantum computing breakthroughs in 2025?"
for s in app.stream({"query": query3, "steps": []}):
    print(s)

print("\n--- Running Example 4: Query with limited internal KB info ---")
query4 = "Tell me about the history of deep learning frameworks."
for s in app.stream({"query": query4, "steps": []}):
    print(s)


### Interpreting the Output and Performance Trade-offs

The code demonstrates a flexible RAG system that dynamically adapts its information retrieval strategy. When you run the examples, observe the `---NODE: ...---` print statements. These indicate the path the agent takes through the LangGraph workflow:

*   **Example 1 (LangGraph/Agentic RAG)**: The `retrieve_from_internal_kb` tool provides a relevant answer. The `decide_to_search_node` (powered by the LLM) determines that the internal RAG result is sufficient, and the flow proceeds directly to `generate_answer` without invoking `web_search`.
*   **Example 2 (Latest AI Regulations)**: The `retrieve_from_internal_kb` tool indicates its information is outdated. The `decide_to_search_node` correctly identifies the need for more current data and routes the execution to `web_search_node`. After the web search, `generate_answer_node` synthesizes both (or primarily web search) to provide an up-to-date response.
*   **Example 3 (Quantum Computing Breakthroughs)**: Similar to Example 2, the internal KB is insufficient for future/very recent events, triggering a web search.
*   **Example 4 (History of Deep Learning)**: The internal KB provides a generic "no highly relevant documents" message. The `decide_to_search_node` then triggers a web search to find more comprehensive information.

This dynamic routing is the core of agentic behavior, allowing the system to be more robust and comprehensive than a static RAG system.

#### Performance Trade-offs:

1.  **Accuracy and Freshness (Pro)**: The most significant advantage is the ability to access up-to-date and broader information, drastically improving answer accuracy and relevance, especially for dynamic topics or when the internal KB is limited.
2.  **Latency (Con)**: Performing a web search introduces additional latency. While modern search APIs are fast, they are still slower than querying an optimized vector database locally. For applications requiring extremely low-latency responses, this needs careful consideration.
3.  **Cost (Con)**: External API calls (LLMs for decision-making and synthesis, and web search APIs) incur costs. A purely internal RAG system might be cheaper if its knowledge base is sufficient.
4.  **Complexity (Con)**: The LangGraph setup, while powerful, adds complexity compared to a linear RAG chain. Designing effective prompts for the `decide_to_search` and `generate_answer` nodes is crucial for optimal performance.
5.  **Information Overload/Quality (Potential Con)**: Web search can return a vast amount of information, not all of which is high quality or directly relevant. The LLM's ability to filter, synthesize, and discern reliable sources becomes paramount.
6.  **Robustness (Pro)**: The system becomes more resilient to gaps in its internal knowledge, making it suitable for a wider range of queries and use cases.

#### Typical Use Cases:

*   **Dynamic Q&A Systems**: For customer support or internal knowledge bases that need to answer questions about rapidly changing topics (e.g., market trends, product updates, regulatory changes).
*   **Research Assistants**: Aiding researchers by combining internal documents with the latest findings from the web.
*   **General Purpose Chatbots**: Enhancing chatbots to handle a broader spectrum of user queries beyond their pre-programmed or pre-indexed knowledge.
*   **Competitive Intelligence**: Monitoring competitor activities or industry news that might not be in internal reports yet.

By intelligently integrating web search, agentic RAG systems move beyond mere retrieval to become truly intelligent information agents, capable of navigating the vast landscape of human knowledge.


### Resources

*   **LangGraph Documentation**: The official source for building stateful, multi-actor applications with LLMs. Explore advanced graph patterns and state management.
    *   [https://langchain-ai.github.io/langgraph/](https://langchain-ai.github.io/langgraph/)
*   **LangChain Expression Language (LCEL)**: Understand the building blocks for creating robust and composable LLM applications.
    *   [https://python.langchain.com/docs/expression_language/](https://python.langchain.com/docs/expression_language/)
*   **Google AI Studio / Gemini API**: For powerful and cost-effective LLMs, especially Gemini 1.5 Flash/Pro.
    *   [https://ai.google.dev/](https://ai.google.dev/)
*   **OpenAI API**: Access to state-of-the-art models like GPT-4o for advanced reasoning and generation.
    *   [https://platform.openai.com/](https://platform.openai.com/)
*   **Tavily Search API**: A popular choice for integrating real-time web search into LLM applications.
    *   [https://tavily.com/](https://tavily.com/)
*   **Brave Search API**: An alternative privacy-focused web search API for developers.
    *   [https://brave.com/search/api/](https://brave.com/search/api/)
